In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

In [ ]:
config_dir = "/storage/project/r-jmarkowitz30-0/jmarkowitz30/dev/python/depth_keys/example_configs/default"

config_path = os.path.join(config_dir, "config.toml")
skeleton_path = os.path.join(config_dir, "skeleton.json")
transforms_path = os.path.join(config_dir, "avg_transforms.json")

centroid_model_path = "/storage/home/hcoda1/3/triesenmy3/lab_folder/sleap_nn_models/models/centroid_unet"
ci_model_path = "/storage/home/hcoda1/3/triesenmy3/lab_folder/sleap_nn_models/models/convnext-large_seed-4"
intrinsics_path = "/storage/project/r-jmarkowitz30-0/shared/active_lab_members/markowitz_jeffrey/active_projects/mouse_open_field_lucid_rig_da_photometry/intrinsics_lucid_rig.toml"
reference_camera = "Lucid Vision Labs-HTP003S-001-224500508"
conda_env_name = None
version_num = 1
cable = False

In [33]:
# sys.path.insert(0, str(REPO_ROOT))

node_names = [
    "tail_tip",
    "tail_middle",
    "tail_base",
    "back_bottom",
    "back_middle_lower",
    "back_middle_upper",
    "back_top",
    "left_ear",
    "right_ear",
    "snout",
    "left_hip",
    "right_hip",
    "left_shoulder",
    "right_shoulder",
]

In [34]:
from depth_keys.experiment.trial import Trial
from glob import glob
import warnings

In [35]:
def process_session(
    source_directory,
    config_path,
    ci_model_path,
    centroid_model_path,
    intrinsics_path,
    transforms_path,
    skeleton_path,
    node_names,
    glob_pattern="_proc/*.avi",
    version_num=1,
    reference_camera="",
    cable=False,
    compute_2d=True,
    compute_3d=True,
    compute_renders=True,
):
    video_paths = sorted(glob(os.path.join(source_directory, glob_pattern)))
    inference_output_path = os.path.join(source_directory, "_proc", f"_key_points_v{version_num}")
    keypoints3d_output_path = os.path.join(source_directory, "_proc", f"_key_points_v{version_num}_3d")
    renders_output_path = os.path.join(source_directory, "_proc", "renders")

    if len(video_paths) > 0:
        print(f"Processing videos in {source_directory}: {video_paths}")

    trial = Trial(
        trial_id=source_directory,
        video_paths=video_paths,
        version_num=version_num,
        base_dir=os.path.dirname(source_directory),
        node_names=NODE_NAMES,
        video_extension=".avi",
        inference_output_path=inference_output_path,
        keypoints_output_path=keypoints3d_output_path,
        reference_camera=reference_camera,
        intrinsics_file=intrinsics_path,
        cable=cable,
        conda_env_name=None,
        transforms_path=transforms_path,
    )

    inference_exists = os.path.exists(inference_output_path)
    keypoints3d_exists = os.path.exists(keypoints3d_output_path)
    renders_exists = os.path.exists(renders_output_path)

    # process_session: 2D keypoint prediction
    if compute_2d and not inference_exists:
        trial.predict_keypoints(ci_model_path=ci_model_path, centroid_model_path=centroid_model_path)
    elif compute_2d and inference_exists:
        warnings.warn(f"Inference directory: {inference_output_path} already exists")
        return None
    # post_process: 2D -> 3D conversion
    if compute_3d and not keypoints3d_exists:
        trial.compute_3d_keypoints(config_path=config_path)
    elif compute_3d and keypoints3d_exists:
        warnings.warn(f"3D directory: {keypoints3d_output_path} already exists")
        return None
    
    # visualize: render keypoint overlay + 3D matplotlib video
    if compute_renders and not renders_exists:
        alt_key_path = os.path.join(trial.keypoints_output_path, "merged_keypoints.h5")
        trial.visualize(
            matplot_viz=True,
            overlay_viz=True,
            output_dir=renders_output_path,
            skeleton_json_path=skeleton_path,
            alt_key_path=alt_key_path,
        )
    elif compute_renders and renders_exists:
        warnings.warn(f"Renders directory: {renders_output_path} already exists")
        return None
    # po

    # os.makedirs(renders_output_path, exist_ok=False)

In [36]:
test_directory = "/storage/project/r-jmarkowitz30-0/shared/active_lab_members/markowitz_jeffrey/active_projects/6-OHDA/session_20260526133956-024733 (system76-pc)"

In [37]:
process_session(
    test_directory,
    config_path=config_path,
    ci_model_path=ci_model_path,
    centroid_model_path=centroid_model_path,
    intrinsics_path=intrinsics_path,
    transforms_path=transforms_path,
    skeleton_path=skeleton_path,
    node_names=node_names,
)

Processing videos in /storage/project/r-jmarkowitz30-0/shared/active_lab_members/markowitz_jeffrey/active_projects/6-OHDA/session_20260526133956-024733 (system76-pc): ['/storage/project/r-jmarkowitz30-0/shared/active_lab_members/markowitz_jeffrey/active_projects/6-OHDA/session_20260526133956-024733 (system76-pc)/_proc/Lucid Vision Labs-HTP003S-001-223702048.avi', '/storage/project/r-jmarkowitz30-0/shared/active_lab_members/markowitz_jeffrey/active_projects/6-OHDA/session_20260526133956-024733 (system76-pc)/_proc/Lucid Vision Labs-HTP003S-001-223702266.avi', '/storage/project/r-jmarkowitz30-0/shared/active_lab_members/markowitz_jeffrey/active_projects/6-OHDA/session_20260526133956-024733 (system76-pc)/_proc/Lucid Vision Labs-HTP003S-001-224500508.avi']


/tmp/ipykernel_2102772/1163334634.py:50: UserWarning: Inference directory: /storage/project/r-jmarkowitz30-0/shared/active_lab_members/markowitz_jeffrey/active_projects/6-OHDA/session_20260526133956-024733 (system76-pc)/_proc/_key_points_v1 already exists
  warnings.warn(f"Inference directory: {inference_output_path} already exists")


In [ ]:
for p in [INFERENCE_OUTPUT_PATH, KEYPOINTS_OUTPUT_PATH, RENDERS_OUTPUT_PATH]:
    print(f'{p.name} -> {"exists" if p.exists() else "missing"}')